# Data Collection with Yelp Fusion API

**In the practice we will use requests+json to develop a simple but efficient Yelp crawler (offical API based)**

In lots of situation, we can get data from popular platforms such as google map, Yelp etc. using their official Web APIs. To use these Web APIs, you will need a developer account for the target platform, and in general, there are QPS and amount limit for the free quota. Of course you can always buy more quota if your want (a little expensive for students).

For google and Yelp's APIs such as Yelp's "Bussiness Search", each request will just return a small amount of data (the entire data is divied into a bunch of pages), and there is a limitation of the total results you can get in a single research. For example, in Seattle, WA, there may be 30000 restuarants, however in a single search your can get no more than 1000 resturants, and the result is divided into 20 pages, each page contains 50 records (one request, one page). To approximate the real number "30,000", you will need advanced techniques such as grid search algorithm or recursive search algorithm. We wouldn't learn about thse advanced techniques in this turtorial, but we will manage to get the allowed max num in one single serach by simply retrieve data in different pages.

### Key Knowledges:
- Workflow of a web crawler
  - send request-> recieve response-> extract needed information from the response->reformat and storage
- Yelp Fusion API
  - the official web APIs of Yelp, you can get Yelp's data using these APIs
  - more info refer to: https://www.yelp.com/developers/documentation/v3/business_search
  - in the practice, we will use two API called "Bussiness Search" and "Comments"
- Request
  - send a request to a server and get the response
  - the content of the response can be json, html, image etc.
- Json
  - A key-value style data format, used widely in transmit web data
  - We use libary json in python to extract needed information from returned data of a web request

by Yifan

In [14]:
import requests
import json
import math
import pandas as pd
import time

# API endpoints
BUSINESS_SEARCH = "https://api.yelp.com/v3/businesses/search"
REVIEWS = "https://api.yelp.com/v3/businesses/{id}/reviews"

# We were going to use Business Details to get store hour information
# Ultimately decided to go a different direction for visual
# BUSINESS_DETAILS = "https://api.yelp.com/v3/businesses/{id}"

# API Key (replace with your own key; DO NOT hard-code in public code)
API_KEY = "UjNgK9LXje1kUwaQMZod9PgKF2lnTuPAvVRxkB1gqWtyyAUvU6T3zw1Gyd1Q_usJrY1n8ZMbuy8VuqWgOnpXeXO3pa9EUCORXDzEjXludoVU7RE0DfOg91hr_-eEaXYx"
HEADER = {"Authorization": f"Bearer {API_KEY}"}

# Yelp Fusion Search API returns up to 240 results for a single query
MAX = 100
LIMIT = 50

def _normalize_price(price):
    """
    Normalize the `price` argument to Yelp API format.

    Accepts:
      - None
      - int (1~4)
      - list/tuple of ints (each 1~4), length <= 4

    Returns:
      - None (if price is None)
      - "1,2,3" string (comma-separated)
    """
    if price is None:
        return None

    if isinstance(price, int):
        price_list = [price]
    else:
        price_list = list(price)

    if len(price_list) > 4 or any((not isinstance(p, int) or p < 1 or p > 4) for p in price_list):
        raise ValueError("price must be an integer 1~4 (or a list/tuple of them), with length <= 4. e.g., price=[1,2,3]")

    return ",".join(map(str, price_list))

def yelp_search(term="bar", location="Seattle, WA", pause=0.25, includeComments=False, price=None):
    """
    Extract business information from Yelp Fusion API.

    Retrieves basic business information (up to MAX items) and optionally attached reviews
    (up to 3 reviews per business in Yelp API).

    Reference:
      https://docs.developer.yelp.com/docs/fusion-intro

    Args:
      term: Search term, e.g., "food", "restaurants", "Starbucks".
      location: Geographic area for searching businesses.
      pause: Sleep time (seconds) between requests to avoid overheating QPS.
      includeComments: If True, fetch up to 3 reviews for each business.
      price: None or a subset of [1,2,3,4], where 1=$, 2=$$, 3=$$$, 4=$$$$.

    Returns:
      A pandas DataFrame.
      - If includeComments=False:
          columns: content_id, name, rating, reviews, phone, address, city, state, country,
                   postcode, latitude, longitude
      - If includeComments=True:
          adds: comment_id, user_id, user_name, user_rating, comment
    """

    # Build query params (IMPORTANT: add price BEFORE requests)
    params = {
        "term": term,
        "location": location,
        "limit": LIMIT,
    }

    price_str = _normalize_price(price)
    if price_str is not None:
        params["price"] = price_str

    total = MAX
    offsets = [i * LIMIT for i in range(math.ceil(total / LIMIT))]

    rows = []
    print("********** START **********")

    for offset in offsets:
        print(f"Retrieving: {offset}/{total}")
        time.sleep(pause)

        params["offset"] = offset
        response = requests.get(BUSINESS_SEARCH, headers=HEADER, params=params)

        # Basic error handling
        if response.status_code != 200:
            raise RuntimeError(f"Yelp API error {response.status_code}: {response.text}")

        payload = response.json()
        businesses = payload.get("businesses", [])

        # If API returns an empty page, stop early (saves quota)
        if not businesses:
            break

        for item in businesses:
            content_id = item.get("id")

            name = item.get("name")
            rating = item.get("rating")
            reviews = item.get("review_count")
            phone = item.get("phone")

            loc = item.get("location", {}) or {}
            address = loc.get("address1")
            city = loc.get("city")
            state = loc.get("state")
            country = loc.get("country")
            postcode = loc.get("zip_code")

            coord = item.get("coordinates", {}) or {}
            latitude = coord.get("latitude")
            longitude = coord.get("longitude")

            rows.append([
                content_id, name, rating, reviews, phone, address, city, state,
                country, postcode, latitude, longitude
            ])

    df = pd.DataFrame(
        rows,
        columns=[
            "content_id", "name", "rating", "reviews", "phone", "address",
            "city", "state", "country", "postcode", "latitude", "longitude"
        ]
    )

    df.drop_duplicates(subset="content_id", keep="first", inplace=True)
    print(f"{len(df)} {term} at {location} retrieved.")

    # Fetch additional reviews (optional)
    if includeComments and len(df) > 0:
        review_rows = []
        for i, content_id in enumerate(df["content_id"], start=1):
            if i % 5 == 0:
                print(f"Reviews retrieving: {i}/{len(df['content_id'])}")
            time.sleep(pause)

            r = requests.get(REVIEWS.format(id=content_id), headers=HEADER)
            if r.status_code != 200:
                raise RuntimeError(f"Yelp Reviews API error {r.status_code}: {r.text}")

            reviews_payload = r.json()
            reviews_list = reviews_payload.get("reviews", [])

            for rev in reviews_list:
                comment_id = rev.get("id")
                user = rev.get("user", {}) or {}
                user_id = user.get("id")
                user_name = user.get("name")
                user_rating = rev.get("rating")
                comment = rev.get("text")
                review_rows.append([content_id, comment_id, user_id, user_name, user_rating, comment])

        cdf = pd.DataFrame(
            review_rows,
            columns=["content_id", "comment_id", "user_id", "user_name", "user_rating", "comment"]
        )

        print(f"{len(cdf)} attached reviews retrieved.")
        df = pd.merge(df, cdf, on="content_id", how="left")

    print("********** FINISH **********")
    return df

## The following code cell will return 100 businesses and costomer reviews. Revise search term and location accordingly and excute the code cell.

**Note**
How many API requests do I receive?


**Update 7/5/2024**: Yelp Fusion API keys registered as Starter, 2024 receive 300 API calls per day by default. Plan well with your search!



You can change 'term', 'location', and price.

* term: Search term that you actually write down in Yelp
* location: Region name, i.e. city, neighborhood and etc.
* price: level of expensiveness. 1 is the cheapest and the 4 is the most expensive. You can search multiple level by using list. For example, you can not only use `price = 1`, but also `price = [1, 2]`, `price = [1, 2, 3, 4]` and etc.




In [15]:
#Excute Demo: Extract basic information of Yelp search result

MAX=10 # Just for demo to reduce API usage. Please delete this line in real practice, or set "MAX=100".
df1=yelp_search(term='cafe',location='Fremont, Seattle, WA', pause=0.25, includeComments=False, price = [1,2])
df1

********** START **********
Retrieving: 0/10
50 cafe at Fremont, Seattle, WA retrieved.
********** FINISH **********


,content_id,name,rating,reviews,phone,address,city,state,country,postcode,latitude,longitude
0,3jrQdJz7YPGOi6KbDLmnTg,Aroom Coffee,4.6,495,,3801 Stone Way N,Seattle,WA,US,98103,47.653220,-122.342833
1,H_lQTUTQm9K0tmEJLZ4LAQ,Milstead,4.3,555,+12066594814,754 N 34th St,Seattle,WA,US,98103,47.649389,-122.347660
2,f8gSWLWxpgoFeCdPS-1aiQ,Fremont Coffee,3.9,481,+12066323633,459 N 36th St,Seattle,WA,US,98103,47.651670,-122.353050
3,xW8aXxMdmqbRqhE_noP1RQ,Stone Way Cafe,4.2,479,+12064204435,3525 Stone Way N,Seattle,WA,US,98103,47.650306,-122.343063
4,VFVaRMjj00G3vnSq0uMfvg,Lighthouse Roasters,4.3,354,+12066334775,400 N 43rd St,Seattle,WA,US,98103,47.659111,-122.354164
5,BRbt5hU92wntHuYdxTWICg,Valentina’s Cafe,4.4,102,,5405 Leary Ave NW,Seattle,WA,US,98107,47.666870,-122.382820
6,6CxoKzPerEnqthyNwdS76w,Seattle Meowtropolitan,4.5,488,+12066322330,1225 N 45th St,Seattle,WA,US,98103,47.661250,-122.342910
7,1x9sa8uepYXjmTMGSrZqQQ,Caffe Ladro,3.9,208,+12066750854,452 N 36th St,Seattle,WA,US,98103,47.652240,-122.353080
8,8doly0Yc4kGgOsmd7EjBAA,Red Arrow Coffee,4.6,74,,425 NW Market St,Seattle,WA,US,98107,47.668519,-122.363074
9,1o2LDytKCmW9PVcYh5q7kw,Sabine Cafe and Bar,4.1,395,+12062080281,5307 Ballard Ave NW,Seattle,WA,US,98107,47.666440,-122.383660


Now the following code will filter the scraped data and store the columns that we need in the next step into a new data frame with myData.

In [16]:
myData=df1.filter(items=['name','rating','reviews', 'latitude', 'longitude'])
myData

,name,rating,reviews,latitude,longitude
0,Aroom Coffee,4.6,495,47.653220,-122.342833
1,Milstead,4.3,555,47.649389,-122.347660
2,Fremont Coffee,3.9,481,47.651670,-122.353050
3,Stone Way Cafe,4.2,479,47.650306,-122.343063
4,Lighthouse Roasters,4.3,354,47.659111,-122.354164
5,Valentina’s Cafe,4.4,102,47.666870,-122.382820
6,Seattle Meowtropolitan,4.5,488,47.661250,-122.342910
7,Caffe Ladro,3.9,208,47.652240,-122.353080
8,Red Arrow Coffee,4.6,74,47.668519,-122.363074
9,Sabine Cafe and Bar,4.1,395,47.666440,-122.383660


# **Now you will download data in CSV and clean.**







The folloing code cells will save pandas df in csv into your google drive.  

In [17]:
# Import Drive API and authenticate.
from google.colab import drive

# Mount your Drive to the Colab VM.
drive.mount('/content/drive')

# Write the DataFrame to CSV file.
# Check the file path
with open('/content/drive/MyDrive/Colab Notebooks/myYelpData.csv', 'w') as f:
  myData.to_csv(f)

Mounted at /content/drive


In [18]:
# Download csv in your local drive.
# Check the file path.
from google.colab import files
#files.download('/content/drive/MyDrive/Colab Notebooks/FremontYelpData.csv')
files.download('/content/drive/MyDrive/Colab Notebooks/myYelpData.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Our Data Retrievals for Ballard, Seattle**

Above was mostly a demo. Will copy the yelp_search code and the file management code here, start querying Yelp for data on sites hopefully near Ballard and the area we walked.

In [19]:
MAX=100
ballard_breakfast = yelp_search(term='breakfast',location='Ballard, Seattle, WA', pause=0.25, includeComments=False) # omitted price = []
#ballard_cafe
# rough check of the locations with Google Maps, seems like mix of Ballard and non-Ballard locations
# might be able to exclude options outside a certain (lat,long) range

********** START **********
Retrieving: 0/100
Retrieving: 50/100
100 breakfast at Ballard, Seattle, WA retrieved.
********** FINISH **********


In [20]:
breakfast = ballard_breakfast.filter(items=['name','rating','reviews', 'latitude', 'longitude'])
breakfast

,name,rating,reviews,latitude,longitude
0,Sabine Cafe and Bar,4.1,395,47.666440,-122.383660
1,A Lamb’s Quandary,4.6,31,47.679680,-122.374430
2,Rachel's Bagels & Burritos,4.2,726,47.667830,-122.383980
3,The Block Cafe,4.2,13,47.662465,-122.374454
4,Cloudy Cafe,4.5,17,47.675695,-122.387304
...,...,...,...,...,...
95,Mojito,4.8,31,47.690795,-122.317931
96,El Taco Loco,4.8,99,47.668797,-122.394993
97,Union Saloon,4.4,137,47.651897,-122.336478
98,Copine,4.5,266,47.675698,-122.387260


In [21]:
# Import Drive API and authenticate.
from google.colab import drive

# Mount your Drive to the Colab VM.
drive.mount('/content/drive')

# Writing dataframe to csv, and checking path
with open('/content/drive/MyDrive/Colab Notebooks/ballardBreakfast.csv', 'w') as f:
  #ballard_cafe.to_csv(f)
  breakfast.to_csv(f)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
from google.colab import files
files.download('/content/drive/MyDrive/Colab Notebooks/ballardBreakfast.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
ballard_dinner = yelp_search(term='dinner',location='Ballard, Seattle, WA', pause=0.25, includeComments=False) # omitted price = []
dinners = ballard_dinner.filter(items=['name','rating','reviews', 'latitude', 'longitude'])
dinners

********** START **********
Retrieving: 0/100
Retrieving: 50/100
100 dinner at Ballard, Seattle, WA retrieved.
********** FINISH **********


,name,rating,reviews,latitude,longitude
0,Andalucia Restaurant,4.5,6,47.666927,-122.384132
1,The Ballard Cut,4.5,310,47.668504,-122.385847
2,Asadero Ballard,4.4,1524,47.666870,-122.382820
3,Rupee Bar,4.3,228,47.674830,-122.387850
4,The Little Beast,4.2,28,47.664463,-122.381379
...,...,...,...,...,...
95,Moon Tree Sushi And Tapas,4.6,520,47.623928,-122.355137
96,Dreamland Bar & Diner,4.5,338,47.650109,-122.351620
97,Green Tree,4.5,291,47.692217,-122.355083
98,Situ Tacos,3.8,90,47.666267,-122.383231


In [24]:
with open('/content/drive/MyDrive/Colab Notebooks/ballardDinner.csv', 'w') as f:
  dinners.to_csv(f)

In [25]:
files.download('/content/drive/MyDrive/Colab Notebooks/ballardDinner.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
ballard_lunch = yelp_search(term = "lunch", location='Ballard, Seattle, WA', pause=0.25, includeComments=False) # omitted price = []
lunch = ballard_lunch.filter(items=['name','rating','reviews', 'latitude', 'longitude'])
lunch

********** START **********
Retrieving: 0/100
Retrieving: 50/100
100 lunch at Ballard, Seattle, WA retrieved.
********** FINISH **********


,name,rating,reviews,latitude,longitude
0,Sabine Cafe and Bar,4.1,395,47.666440,-122.383660
1,Un Bien,4.7,1551,47.681689,-122.376638
2,Nirvana,4.9,46,47.668023,-122.379152
3,Cloudy Cafe,4.5,17,47.675695,-122.387304
4,Asadero Ballard,4.4,1524,47.666870,-122.382820
...,...,...,...,...,...
95,Esters Enoteca,4.5,118,47.650100,-122.349576
96,Old Salt Fish and Bagels,4.1,32,47.674870,-122.387840
97,Staple & Fancy,4.1,830,47.663540,-122.380070
98,SWeL Restaurant,4.6,495,47.650669,-122.351732


In [27]:
with open('/content/drive/MyDrive/Colab Notebooks/ballardLunch.csv', 'w') as f:
  lunch.to_csv(f)

In [28]:
files.download('/content/drive/MyDrive/Colab Notebooks/ballardLunch.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **From Crawled Tables to Hierarchical Data (Team Workflow)**

Before going to the next step, you will clean and edit your csv file manually in Excel. **Note that the crawler does not generate a type column by default.** This is intentional. In this step, you will add two additional column to build a context and record data collector's name.

This assignment is designed as a **team-based workflow**, where each team member conducts independent searches using different criteria and creates their own CSV file. After crawling, each team member should manually **add a `type` column** to their CSV, indicating the category of the search they performed.

The `type` field does not have to be limited to business types such as Cafe, Restaurant, or Bar. It may also reflect alternative grouping logics, such as price range (1~4), search keywords, or any other criteria used during data collection.

**Each team member also add a `developer` column and record their full name** before merging files.

Let's check again. Does your table has columns of 'name','type', 'rating' (or an another numeric value column), 'lat', 'lon', and 'developer'? Yes? good! You are ready to merge them into one table. Combine each tables to one.

Once all individual CSV files are combined into a single dataset, the merged file should be uploaded and processed using the code above to generate the hierarchical JSON structure.

You will submit the merged CSV file for the data collection part and move to the next step. You are doing great!

# **Build a nested database**

If you finish editing and merging your group's csv files, let's convert `.csv` file into `.json` file.

To visualize the radial dendrogram, it is important to understand the structure of 'nested' data.

Below are the concept of 'nested data' and the code that actually enables to convert csv to nested json file.

## Way 1.
The following code constructs a three-level hierarchical JSON structure from a tabular business dataset.

*  At the **root level**, the data are organized by Neighborhood.
*  At the **group level**, businesses are grouped by Type (e.g., Cafe, Restaurant, Bar).
*  At the **leaf level**, each node represents an individual business and contains attributes such as geographic coordinates (latitude and longitude), rating, or review count.

**Update Jan/29/2026**
you have to revise the code line with your second tier term. If you use 'rating' instead of 'type' at the group level, then revise the example conversion code properly to reflect the hierarchy in your data.

![tree_graph](https://drive.google.com/uc?id=1E-YtNf565nYWkmkbGooel7iZMjC0xFM6)

Image credit: Se Chang Kim

In [29]:
# test 1/29/2026
def build_root_node(df_all, root_name="Ballard"): # Change Neighborhood Name
    return {
        "name": root_name,
        "children": list(build_group_nodes(df_all))
    }

def build_group_nodes(df_all):
    for (group_value,), df_group in df_all.groupby(["type"], dropna=False):
        yield {
            "name": group_value,
            "children": list(build_leaf_nodes(df_group))
        }

def build_leaf_nodes(df_group):
    for row in df_group.itertuples(index=False):
        yield {
            "name": row.name,
            "value": row.rating
        }

Instead of passing a data frame directly from the crawler, you will now upload your cleaned CSV file, read it as a pandas DataFrame, and pass it into the method defined above.

As a result, the output cell will display a nested JSON structure similar to the example shown below. You will be able to observe a **single root node representing a neighborhood**, an **indented middle-tier node representing business types**, and a **further indented leaf node representing individual stores**.

This three-level hierarchy (neighborhood → type → store) reflects how the data are organized for the radial dendrogram visualization.

![json_example](https://drive.google.com/uc?id=1mWahqKykTFD6V9j0UhGr6NaOQbh4llBF)

In [30]:
# when pd is not defined, import pandas again
import pandas as pd

# with NameError: name 'json' is not defined, import json
import json

Check if myYelpData has created successfully.

In [31]:
df_pd = pd.read_csv('/content/sample_data/BallardYelp.csv') # replace myData with the file path to your csv like pd.read_csv('/content/drive/MyDrive/495/data/myData.csv')

review_tree = build_root_node(df_pd)
myYelpData = json.dumps(review_tree, indent=4)

In [32]:
print(myYelpData)

{
    "name": "Ballard",
    "children": [
        {
            "name": "allday",
            "children": [
                {
                    "name": "Brimmer & Heeltap",
                    "value": 4.3
                },
                {
                    "name": "El Moose",
                    "value": 4.0
                },
                {
                    "name": "Freya",
                    "value": 4.5
                },
                {
                    "name": "Mainstay Provisions",
                    "value": 4.3
                },
                {
                    "name": "Ocho",
                    "value": 4.0
                },
                {
                    "name": "Rachel's Bagels & Burritos",
                    "value": 4.2
                },
                {
                    "name": "S/T Hooligans",
                    "value": 4.7
                },
                {
                    "name": "Sabine Cafe and Bar",
                

Our data is ready to store in an physical form and download it.

In [33]:
from google.colab import files

with open('myYelpData.json', 'w') as f:
  f.write(myYelpData)
  files.download('myYelpData.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**myYelpData**.json should be in your download folder. Open it in a text editor and begin to develop a context of your neighborhood story.

## Way 2.
There is another way. If you're more familiar with pandas and for loop, you can follow this way.
First, upload the csv file.

In [ ]:
import json
from google.colab import files

files.upload() # upload the csv file

TypeError: 'NoneType' object is not subscriptable

Next, run the code below. The json file will be created in this notebook.
Make sure you should match the 'column' name!

In [ ]:
# read csv
df = pd.read_csv("BallardYelp.csv")

# container for middle-tier nodes
type_nodes = []

# group by 'type' -> make group and leaf node
for t, g in df.groupby("type"):
    children = []
    for _, row in g.iterrows():
        children.append({
            "name": row["name"],
            "value": row["rating"]
        })

    type_nodes.append({
        "name": t,
        "children": children
    })

# root node
nested_json = {
    "name": "root",
    "children": type_nodes
}

# save to json
with open("output.json", "w", encoding="utf-8") as f:
    json.dump(nested_json, f, ensure_ascii=False, indent=2)

If you have any questions about the code, feel free to ask!

Also, when you finish convert csv file into json file, now it's time to go to observable and finally visualize with radial dendrogram! **Go to Canvas "Let's Code at Observable" page.**

Yelp Crawler code updated on Oct. 21. 2022.  
A nested json conversion code updated on July 10, 2023   
The instruction was updated on Jan. 30, 2026 by Se Chang Kim. Huge thanks to him!